# Lab 1: Simple Object Detection and Benchmarking with OpenVINO

**Goals:**
- Run object detection using OpenVINO
- Understand inference flow
- Measure throughput and FPS

## Installation

Run the cell below once to install required libraries.

In [ ]:
# Install required libraries (run once)
!pip install openvino opencv-python numpy matplotlib ipywidgets

## Simple Pipeline

**Input Image** → **Preprocessing** → **OpenVINO Model** → **Detection Output** → **Bounding Boxes**

## Setup

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import openvino as ov
import time
import os
import glob
from ipywidgets import Dropdown
from IPython.display import display

# Configuration
IMAGE_PATH = "media/sample_image-1.jpg"
PRECISION = "FP16"

model_dropdown = Dropdown(options=["ATSS-MobileNetV2", "Deim-DFine-X"], value="ATSS-MobileNetV2", description="Model:")
device_dropdown = Dropdown(options=["CPU", "GPU", "NPU"], value="CPU", description="Device:")
display(model_dropdown, device_dropdown)

In [ ]:
# Create dummy image if not present (for local testing)
os.makedirs("media", exist_ok=True)
if not os.path.exists(IMAGE_PATH):
    # Synthetic road-like image (640x480) similar to sample_image-1.jpg
    h, w = 480, 640
    img = np.zeros((h, w, 3), dtype=np.uint8)
    img[:] = (90, 90, 95)  # Gray road
    cv2.rectangle(img, (100, 200), (250, 350), (40, 40, 40), -1)   # Dark car
    cv2.rectangle(img, (350, 220), (500, 360), (180, 180, 180), -1)  # Light car
    cv2.rectangle(img, (200, 280), (320, 400), (0, 0, 150), -1)     # Red car
    cv2.imwrite(IMAGE_PATH, img)
    print(f"Created dummy image: {IMAGE_PATH}")

## Model Loading

In [ ]:
def find_model_xml(path):
    """Find .xml file in the given path."""
    files = glob.glob(os.path.join(path, "*.xml"))
    if not files:
        raise FileNotFoundError(f"No .xml model found in {path}")
    return files[0]

MODEL_NAME = model_dropdown.value
model_dir = os.path.join("models", MODEL_NAME, PRECISION)
xml_path = find_model_xml(model_dir)

core = ov.Core()
model = core.read_model(xml_path)
compiled_model = core.compile_model(model, device_dropdown.value)

input_layer = compiled_model.input(0)
input_shape = input_layer.shape
print(f"Model loaded: {xml_path}")
print(f"Device: {device_dropdown.value}")
print(f"Input shape: {input_shape}")

## Image Inference

In [ ]:
def preprocess_image(img, input_shape):
    """Resize and convert to NCHW format."""
    _, _, h, w = input_shape
    resized = cv2.resize(img, (w, h))
    nchw = np.expand_dims(resized.transpose(2, 0, 1), axis=0).astype(np.float32)
    return nchw

def run_inference(compiled_model, input_tensor):
    """Run synchronous inference."""
    return compiled_model(input_tensor)

def postprocess(output, orig_h, orig_w, model_name, thresh=0.5):
    """Parse detection output to boxes [x1, y1, x2, y2]. Supports DetectionOutput (1,1,N,7) and [x1,y1,x2,y2,conf] formats."""
    result = list(output.values())[0]
    arr = np.squeeze(result)
    if arr.ndim == 1:
        arr = np.expand_dims(arr, 0)
    boxes = []
    # DetectionOutput format: [batch_id, class_id, conf, x_min, y_min, x_max, y_max]; batch_id=-1 = end
    if arr.shape[-1] == 7:
        for det in arr:
            batch_id, class_id, conf = float(det[0]), int(det[1]), float(det[2])
            if batch_id < 0:
                break
            if conf <= thresh:
                continue
            x1, y1, x2, y2 = det[3], det[4], det[5], det[6]
            if x2 <= 1 and y2 <= 1:
                x1, y1, x2, y2 = x1*orig_w, y1*orig_h, x2*orig_w, y2*orig_h
            boxes.append([int(x1), int(y1), int(x2), int(y2)])
    else:
        for det in arr:
            if len(det) >= 5 and float(det[4]) > thresh:
                x1, y1, x2, y2 = det[0], det[1], det[2], det[3]
                if x2 <= 1 and y2 <= 1:
                    x1, y1, x2, y2 = x1*orig_w, y1*orig_h, x2*orig_w, y2*orig_h
                boxes.append([int(x1), int(y1), int(x2), int(y2)])
    return np.array(boxes) if boxes else np.zeros((0, 4))

def draw_boxes(img, boxes, color=(0, 200, 0), thickness=2):
    """Draw bounding boxes on image."""
    out = img.copy()
    for box in boxes:
        x1, y1, x2, y2 = box
        cv2.rectangle(out, (x1, y1), (x2, y2), color, thickness)
    return out

In [ ]:
image = cv2.imread(IMAGE_PATH)
if image is None:
    raise FileNotFoundError(f"Image not found: {IMAGE_PATH}")

orig_h, orig_w = image.shape[:2]
input_tensor = preprocess_image(image, input_shape)
output = run_inference(compiled_model, input_tensor)
boxes = postprocess(output, orig_h, orig_w, MODEL_NAME)
result_img = draw_boxes(image, boxes)

plt.figure(figsize=(10, 6))
plt.imshow(cv2.cvtColor(result_img, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title(f"Detections: {len(boxes)} objects")
plt.show()

## Optional: PyTorch to OpenVINO Conversion

For reference only. The main workflow uses pre-converted IR models. Use `convert_model()` and `ov.save_model()` to export PyTorch models to OpenVINO IR.

## Benchmarking: Inference Time by Precision

Single-image object detection. Average inference time (ms) for FP32, FP16, and INT8.

In [ ]:
def benchmark_latency_ms(compiled_model, input_tensor, num_iter=100):
    """Single-image inference; return average latency in ms."""
    for _ in range(10):
        compiled_model(input_tensor)
    times = []
    for _ in range(num_iter):
        start = time.perf_counter()
        compiled_model(input_tensor)
        times.append((time.perf_counter() - start) * 1000)
    return sum(times) / len(times)

results = {}
for prec in ["FP32", "FP16", "INT8"]:
    model_dir_p = os.path.join("models", MODEL_NAME, prec)
    try:
        xml_path_p = find_model_xml(model_dir_p)
        model_p = core.read_model(xml_path_p)
        compiled_p = core.compile_model(model_p, device_dropdown.value)
        lat_ms = benchmark_latency_ms(compiled_p, input_tensor)
        results[prec] = lat_ms
        print(f"{prec} → {lat_ms:.1f} ms")
    except FileNotFoundError:
        print(f"{prec} → (model not found)")
        results[prec] = 0

precisions = [p for p in results if results[p] > 0]
vals = [results[p] for p in precisions]
if precisions:
    plt.figure(figsize=(6, 4))
    plt.bar(precisions, vals, color=["#2ecc71", "#3498db", "#e74c3c"], edgecolor="black")
    plt.xlabel("Precision")
    plt.ylabel("Average Inference Time (ms)")
    plt.title("Average Inference Time by Precision")
    plt.tight_layout()
    plt.show()

## Wrap-up

- OpenVINO simplifies deployment of object detection models
- The detection pipeline is straightforward: load → preprocess → infer → postprocess
- Lower precision (FP16, INT8) typically reduces inference time
- Pre-converted IR models keep the workflow simple and fast